# 环节 08 · 输出头与训练目标（配套 Notebook）

> 配套长文：[环节08-输出头与训练目标详解.md](./环节08-输出头与训练目标详解.md)
> 定位：承接 [环节 04 §5](./环节04-Attention注意力详解.md) 的**走查下半场**——预测分布 → 损失 → 梯度，并**真的用这个梯度把模型训一遍**。纯 Python 标准库，零依赖。

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 走查下半场 | §4 | 复现 Z → 预测分布 → loss |
| §2 损失锚点 | §3.2 | `ln\|V\|` 是"乱猜"的水位线；perplexity = e^loss |
| §3 梯度闭合解 | §4 | `∂L/∂z = P − y`，一格不差 |
| §4 真训练一遍 | §5 | loss 从 2.18 降到 0.007，预测从全错变全对 |
| §5 标签右移与 MTP | §3 / §3.2 | 监督信号怎么构造；多 token 预测多监督了什么 |


## 1. 走查下半场：从 Z 到预测分布与损失（长文 §4）

承接环节 04 §5 算出的 Z（残差后的输出）：

```
Z0 = [2.00000, 0, 0, 0]
Z1 = [0.37754, 1.62246, 0, 0]
Z2 = [0.27407, 0.27407, 1.45186, 0]
```

本例 `W_head = I`（恒等），所以 Z 本身就是 logits。标签 `Y = [b, c, d] = [1, 2, 3]`。


In [ ]:
import math


def softmax(row):
    m = max(row)
    e = [math.exp(v - m) for v in row]
    s = sum(e)
    return [v / s for v in e]


Z = [
    [2.0, 0.0, 0.0, 0.0],
    [0.37754, 1.62246, 0.0, 0.0],
    [0.27407, 0.27407, 1.45186, 0.0],
]
Y = [1, 2, 3]                      # 位置0 预测 b，位置1 预测 c，位置2 预测 d
WORDS = ["a", "b", "c", "d"]

logits = Z                          # W_head = I
probs = [softmax(row) for row in logits]

print("Step 7 · 预测分布（每行 = 该位置对“下一个词”的预测）")
print("        " + "".join(f"{w:>9}" for w in WORDS))
for i, row in enumerate(probs):
    mark = f"   ← 期望 {WORDS[Y[i]]}"
    print(f"位置{i}  " + "".join(f"{p:>9.5f}" for p in row) + mark)

print("\n对照长文 §4：")
print("  行0 [0.71123, 0.09626, 0.09626, 0.09626] ✓")
print("  行1 [0.17113, 0.59424, 0.11732, 0.11732] ✓")
print("  行2 [0.16646, 0.16646, 0.54053, 0.12656] ✓")
print("→ 三个位置全猜错（权重是手调的演示值，不是训出来的）—— 这正是好的起点。")


## 2. 损失：交叉熵与它的"水位线"（长文 §3.2）

```
L = −(1/n)·Σ log P[i][ Y[i] ]
```

固定标签时，求和里其它项全是 0，单位置损失退化成 `−log p_c`（真实那个词的概率有多高）。**`ln|V|` 是"完全乱猜"的水位线**。


In [ ]:
n = len(probs)
losses = [-math.log(probs[i][Y[i]]) for i in range(n)]
loss = sum(losses) / n

print("Step 8 · 训练损失")
for i in range(n):
    print(f"  位置{i}: 标签 {WORDS[Y[i]]}，模型给的概率 {probs[i][Y[i]]:.5f}"
          f" → −log = {losses[i]:.4f}")
print(f"\n  loss = ({' + '.join(f'{v:.4f}' for v in losses)}) / {n} = {loss:.5f}")
print(f"  （长文 §4 写 2.184，一致）")

print(f"\n锚点：|V| = {len(WORDS)}，均匀乱猜的损失 = ln|V| = {math.log(len(WORDS)):.4f}")
print(f"  当前 loss {loss:.4f} > {math.log(len(WORDS)):.4f} → 比乱猜还差"
      f"（手调权重与任务错配）")
print(f"\n困惑度 perplexity = e^loss = {math.exp(loss):.3f}（≈ 在 {math.exp(loss):.1f} 个词里犹豫）")
print("\n−log p 的直觉（长文 §3.2）：")
for p in (1.0, 0.5, 0.1):
    print(f"  真词概率 {p:>4} → −log p = {-math.log(p):.4f}")
print("  → 概率 1.0 损失 0；0.5 约 0.69；0.1 约 2.30。只罚“给正确答案的概率够不够高”。")


## 3. 梯度闭合解（长文 §4 Step 9）

```
∂L / ∂z = P − y            # z 是某位置的 logits，y 是标签 one-hot
```

漂亮到值得背：**预测概率减掉标签**。正确词的 logit 想上升（负梯度），其余被压下去。


In [ ]:
i = 2                                       # 以行 2 为例，标签是 d（index 3）
y_onehot = [1.0 if j == Y[i] else 0.0 for j in range(len(WORDS))]
grad_z = [probs[i][j] - y_onehot[j] for j in range(len(WORDS))]

print(f"行 {i}（标签 = {WORDS[Y[i]]}）")
print(f"  P      = {[round(v, 5) for v in probs[i]]}")
print(f"  y      = {y_onehot}")
print(f"  P − y  = {[round(v, 4) for v in grad_z]}")
print("   （长文写 [0.166, 0.166, 0.541, −0.873]，一致）")
print(f"\n含义：要让 d 的 logit 升 {abs(grad_z[3]):.3f}，其余三个各降一点；")
print("  梯度的和恒为 0（softmax 概率和为 1 的必然结果）。")


## 4. 真的用它训一遍（长文 §5 的算法级闭环）

```
前向 logits = Z·W_head  →  loss  →  梯度 ∂L/∂W_head = Zᵀ·(P − y)/n  →  更新
```

把 `W_head` 从单位矩阵出发，跑 300 步，看 loss 与预测怎么变。


In [ ]:
def matmul(A, B):
    cols = list(zip(*B))
    return [[sum(a * b for a, b in zip(row, col)) for col in cols] for row in A]


def transpose(A):
    return [list(col) for col in zip(*A)]


W_head = [[1.0 if i == j else 0.0 for j in range(4)] for i in range(4)]
lr = 0.5
snapshots = []

for step in range(1, 301):
    logits = matmul(Z, W_head)
    p = [softmax(r) for r in logits]
    cur_loss = sum(-math.log(p[i][Y[i]]) for i in range(n)) / n
    if step in (1, 5, 10, 20, 50, 100, 300):
        snapshots.append((step, cur_loss, [max(range(4), key=lambda j: p[i][j]) for i in range(n)]))
    G = [[0.0] * 4 for _ in range(4)]
    for i in range(n):
        for a in range(4):
            for b in range(4):
                G[a][b] += Z[i][a] * (p[i][b] - (1.0 if b == Y[i] else 0.0))
    for a in range(4):
        for b in range(4):
            W_head[a][b] -= lr * G[a][b] / n

print(f"目标标签 Y = {Y}（即 {[WORDS[t] for t in Y]}）\n")
print(f"{'step':>5} {'loss':>10} {'perplexity':>12}   预测")
print("-" * 48)
for step, cur_loss, pred in snapshots:
    flag = "✓ 全对" if pred == Y else "✗"
    print(f"{step:>5} {cur_loss:>10.5f} {math.exp(cur_loss):>12.3f}   {pred} {flag}")

print("\n→ loss 从 2.18 一路降到 0.007（困惑度 8.9 → 1.0），预测从 [0,1,2] 变成 [1,2,3]。")
print("  这就是“训练”的全部意义：把正确答案的概率顶上去。")
print("\n注意：这里只训了 W_head 这一层的 16 个参数。真实训练里，")
print("  梯度会继续沿 logits → Z → 各层 Block → QKV/FFN/Embedding 一路反传（长文 §5 ④）。")


## 5. 标签右移与多 Token 预测（长文 §3 / §3.1）

**Next-token 的标签就是输入右移一位**——这是"自我监督"的全部秘密：不需要人工标注，语料本身就是答案。


In [ ]:
corpus = ["我", "爱", "北京", "天安门"]
print(f"语料            {corpus}")
print(f"输入（前 n−1）  {corpus[:-1]}")
print(f"标签（后 n−1）  {corpus[1:]}")
print("\n逐位置对齐：")
for i in range(len(corpus) - 1):
    print(f"  看到 {corpus[:i+1]} → 应该预测 {corpus[i+1]}")
print("\n→ 每个位置都在做一次“词表分类”，一次前向拿到 n−1 个监督信号（长文 §3.2）。")
print("  这也是为什么训练能整条并行：Mask 保证位置 i 只看得到前缀（环节 04 §2.3）。")


In [ ]:
# MTP：多 token 预测——不只预测下一个，还预测下下个
K = 3                                        # 预测未来 1~3 个 token
print(f"语料 {corpus}\n")
print(f"{'输入':<12} " + " ".join(f"{'头' + str(k) + '预测':>8}" for k in range(1, K + 1)))
print("-" * 46)
for i in range(len(corpus) - 1):
    row = []
    for k in range(1, K + 1):
        row.append(corpus[i + k] if i + k < len(corpus) else "—")
    print(f"{''.join(corpus[:i+1]):<12} " + " ".join(f"{v:>8}" for v in row))

print("\n→ 单个位置现在有 3 个监督信号（主头 + 2 个辅助头），监督密度是原来的 3 倍；")
print("  代价是多了几个输出头（训练完可卸掉）。DeepSeek-V3 / Meta 的 MTP 走这条路")
print("  （长文 §3 表），顺带还能在推理时做自投机解码（环节 11 §1.6）。")


## 6. 自测（长文 §8）

| 问题 | 本 Notebook 的现场证据 |
|---|---|
| LM Head 输出什么？ | §1：全位置 × 词表大小的 logits（未归一化） |
| loss 怎么算的？ | §2：`−log p_c` 按位置平均，得到 2.1836 |
| 什么算"学得好"？ | §2：越过 `ln\|V\| = 1.386` 才算学到东西 |
| 梯度长什么样？ | §3：`P − y`，正确词想升、其余想降 |
| 训练真的能学会吗？ | §4：300 步把 loss 压到 0.007，预测全对 |
| 标签从哪来？ | §5：语料右移一位，无需人工标注 |
| MTP 多监督了什么？ | §5：每个位置监督未来 K 个 token |

**接续**：[环节 09 · 训练管线](./环节09-训练管线详解.md)（把这一步放大到万亿 token 与分布式集群）
